# 🧰 Agent Harness — a batteries-included agent (Portfolio Analyst demo)

A **harness** wraps a chat client with the scaffolding an agent needs for long, multi-step tasks — so you don't assemble it yourself. One factory call, `create_harness_agent`, gives you:

| Feature | What it adds |
|---------|--------------|
| Function-invocation loop | Automatic multi-tool calling until the task is done |
| Per-service-call persistence | History saved after every model call |
| Compaction | Context-window management for long runs |
| TodoProvider | A todo list to plan against |
| AgentModeProvider | Plan / execute mode tracking (interactive) |
| File memory / File access / Skills | Durable memory, file tools, progressive skills (opt-in) |
| Tool approval | "Don't ask again" + heuristic auto-approval |
| Looping | Re-invoke until a predicate is satisfied (interactive) |
| OpenTelemetry | Built-in tracing |

📖 Docs: [Step 6: Agent Harness](https://learn.microsoft.com/en-us/agent-framework/get-started/harness?pivots=programming-language-python) · Samples: [python/samples/02-agents/harness](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/harness)

## What this demo shows

A **Portfolio Research Analyst** harness agent with three local tools. Given one multi-step request, it **plans → calls the tools for every holding → synthesizes a brief** in a single autonomous pass, then answers a follow-up using the same session (memory).

Everything is **self-contained** (mock data, web search disabled) so it runs the same on a laptop and demos reliably.

> ⚠️ Educational demo only — mock prices/news, not real financial advice.

## Prerequisites
- `az login` (uses `AzureCliCredential`)
- Repo `.env` with a Foundry project endpoint + model deployment
- The repository `.venv` kernel selected

## 1️⃣ Imports and environment

In [1]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path
from random import Random

from agent_framework import create_harness_agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the repo .env.")
print("✅ Imports loaded; project endpoint and model configured (values hidden)")

✅ Imports loaded; project endpoint and model configured (values hidden)


## 2️⃣ Define local tools

Three self-contained tools with mock data. Each prints when the model calls it, so you can watch the harness work the tools during the run. `approval_mode="never_require"` keeps the run non-interactive.

In [2]:
_PORTFOLIO = {"CTSO": 120, "FBRK": 40, "NWND": 75}
_NEWS = {
    "CTSO": ["Contoso beats earnings; cloud revenue up 18%.", "Analysts raise Contoso price target."],
    "FBRK": ["Fabrikam recalls a hardware line; margins pressured.", "Fabrikam CFO resigns unexpectedly."],
    "NWND": ["Northwind signs multi-year logistics contract.", "Northwind expands into EU markets."],
}


@tool(approval_mode="never_require")
def get_portfolio() -> str:
    """Return the client's current stock holdings (symbol -> shares)."""
    print("   🔧 get_portfolio()")
    return ", ".join(f"{s}: {n} shares" for s, n in _PORTFOLIO.items())


@tool(approval_mode="never_require")
def get_stock_quote(symbol: str) -> str:
    """Return the latest price for a stock symbol (demo data)."""
    print(f"   🔧 get_stock_quote({symbol})")
    rng = Random(symbol.upper())  # seed by symbol so demos are repeatable
    return f"{symbol.upper()} ${round(rng.uniform(20, 300), 2)} ({round(rng.uniform(-5, 5), 2)}% today)"


@tool(approval_mode="never_require")
def get_company_news(symbol: str) -> str:
    """Return recent headlines for a stock symbol (demo data)."""
    print(f"   🔧 get_company_news({symbol})")
    return " | ".join(_NEWS.get(symbol.upper(), ["No recent news."]))


print("✅ Tools ready: get_portfolio, get_stock_quote, get_company_news")

✅ Tools ready: get_portfolio, get_stock_quote, get_company_news


## 3️⃣ Create the harness agent

`create_harness_agent` turns a chat client into a batteries-included agent. Here we keep the **function-invocation loop, per-service-call persistence, and compaction**, add our tools, and disable the interactive/opt-in features (web search, file memory, plan/execute mode) so a single `run()` completes autonomously. `store=False` lets the harness own conversation history.

In [7]:
INSTRUCTIONS = (
    "You are a portfolio research analyst. First outline a short numbered plan, then carry it out "
    "in the SAME response without waiting for confirmation. For each holding call get_stock_quote and "
    "get_company_news, then recommend ONE position to trim and ONE to add, each with a one-line "
    "rationale. Finish with a short brief. Do not ask clarifying questions; use reasonable defaults. "
    "This is a demo, not real financial advice."
)

# Kept alive across cells; closed in the final cleanup cell.
chat_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=AzureCliCredential(),
)

agent = create_harness_agent(
    client=chat_client,
    name="PortfolioAnalyst",
    agent_instructions=INSTRUCTIONS,
    tools=[get_portfolio, get_stock_quote, get_company_news],
    max_context_window_tokens=128_000,  # enables compaction
    max_output_tokens=8_192,
    disable_web_search=True,      # self-contained demo
    disable_file_memory=True,
    disable_mode=True,            # non-interactive: don't pause for plan/execute confirmation
    default_options={"store": False},
)

# One session carries history (and memory) across turns.
session = agent.create_session()
print("✅ Harness agent 'PortfolioAnalyst' ready")

✅ Harness agent 'PortfolioAnalyst' ready


## 4️⃣ Run a multi-step task (streaming)

Watch the harness **plan**, then call the tools for every holding (🔧 lines), then **synthesize** a brief — all from one request.

In [6]:
task = (
    "Review my portfolio: for each holding, check the latest quote and recent news, "
    "then recommend one position to trim and one to add. Summarize as a short brief."
)

print("💼 PORTFOLIO REVIEW\n" + "=" * 60)
async for chunk in agent.run(task, session=session, stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print("\n" + "=" * 60)

💼 PORTFOLIO REVIEW
   🔧 get_stock_quote(CTSO)   🔧 get_company_news(CTSO)
   🔧 get_stock_quote(FBRK)
   🔧 get_company_news(FBRK)
   🔧 get_company_news(NWND)

   🔧 get_stock_quote(NWND)
Here’s your updated portfolio review:

1. CTSO (120 shares): $217.21 (-4.73% today)  
   - News: Strong earnings beat, cloud revenue up, analysts raising targets.

2. FBRK (40 shares): $239.62 (+2.7% today)  
   - News: Hardware recall and margin pressure; unexpected CFO resignation.

3. NWND (75 shares): $299.52 (-1.1% today)  
   - News: Major contract win, business expansion into Europe.

Recommendation:
- Trim: FBRK—Negative news (recall, C-suite turmoil) increases risk and clouds near-term outlook.
- Add: NWND—New contracts and geographic expansion signal accelerating growth potential.

Brief:
Your portfolio has strong prospects in CTSO and NWND due to positive performance and growth news. FBRK faces risks from operational and leadership turbulence, so consider trimming it and redeploying toward NWND

## 5️⃣ Follow-up on the same session (memory)

The harness persists history per session, so a follow-up understands "the one you told me to trim" without repeating context.

In [5]:
follow_up = "For the position you told me to trim, re-check its quote and explain the risk in 2 bullets."

print("🔁 FOLLOW-UP\n" + "=" * 60)
async for chunk in agent.run(follow_up, session=session, stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print("\n" + "=" * 60)

🔁 FOLLOW-UP
   🔧 get_company_news(FBRK)
   🔧 get_stock_quote(FBRK)
FBRK latest quote: $239.62 (up 2.7% today).

Risks:
- Operational: The recent hardware recall signals potential manufacturing or quality control issues, directly impacting profit margins and possibly damaging brand reputation.
- Leadership: The sudden resignation of the CFO raises concerns about financial oversight and may indicate deeper internal instability or looming strategic challenges.

These factors heighten uncertainty around FBRK’s near-term outlook, justifying caution or a trim in your position.


## 6️⃣ Cleanup

In [ ]:
await chat_client.client.close()
await chat_client.project_client.close()
print("✅ Client closed")

## 📝 Key takeaways

| Concept | Description |
|---------|-------------|
| `create_harness_agent(client, ...)` | One call → function-invocation loop, persistence, compaction, planning, tool approval |
| `tools=[...]` | Your local `@tool` functions the harness calls automatically |
| `agent.run(task, session=..., stream=True)` | The harness plans + works through the whole task in one pass |
| Same `session` across turns | History/memory persists — follow-ups keep context |
| `disable_*` flags | Turn off opt-in features (web search, file memory, mode) to keep a run self-contained |

### Want the full interactive experience?
The official samples add a **Textual console** with plan/execute **modes**, a live **todo list**, and `/todos` `/mode` `/exit` commands. Keep `disable_mode=False` and drive it from a chat loop, or run the official console sample:

- [harness_research.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/02-agents/harness/harness_research.py) — research assistant with planning + web search
- [build_your_own_claw](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/harness/build_your_own_claw) — step-by-step finance agent
- [Harness console package](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/harness/console)

### Next steps
- Add a real data tool (market API, database) in place of the mock functions.
- Give the agent a `memory_store` for durable, cross-session memory.
- Enable `shell_executor` or `skills_paths` for code execution / progressive skills.